# Tutorial 4: Polynomial Arithmetic - Combining Polynomials

## Introduction

In Tutorial 3, we created a general `Polynomial` class that can represent any degree polynomial using coefficient lists. We discovered:

**What we gained:**
- Can represent ANY polynomial degree
- Derivatives and integrals work for everything
- Only ONE class instead of infinite
- No code duplication

**What we lost:**
- Easy exact root formulas (like the quadratic formula)
- Special methods (`vertex()`, `discriminant()`)
- Intuitive interface for common cases

Today, we'll explore what we can **DO** with polynomials:
- **Add** them together
- **Subtract** them
- **Multiply** them
- **Compose** them (substitute one into another)
- **Test equality**

This will show us the full power of our general representation, and set us up for Tutorial 5 where we'll use **inheritance** to get the best of both worlds.

Let's start!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Part A: Starting with Our Complete Polynomial Class

Let's bring in the `Polynomial` class we built in Tutorial 3:

In [ ]:
class Polynomial:
    """
    A class representing a polynomial of any degree.
    
    Attributes:
    -----------
    coeffs : list of float
        Coefficients where coeffs[i] is the coefficient of x^i
    """
    
    def __init__(self, coefficients: list):
        self.coeffs = coefficients
    
    def degree(self) -> int:
        """Return the degree of the polynomial."""
        for i in range(len(self.coeffs) - 1, -1, -1):
            if self.coeffs[i] != 0:
                return i
        return 0
    
    def evaluate(self, x: float) -> float:
        """Evaluate the polynomial at x."""
        result = 0
        for i, coeff in enumerate(self.coeffs):
            result += coeff * (x ** i)
        return result
    
    def __str__(self) -> str:
        """Return a readable string representation."""
        terms = []
        for i, coeff in enumerate(self.coeffs):
            if coeff == 0:
                continue
            if i == 0:
                terms.append(f"{coeff}")
            elif i == 1:
                terms.append(f"{coeff}x")
            else:
                terms.append(f"{coeff}x^{i}")
        if not terms:
            return "0"
        return " + ".join(terms)
    
    def plot(self, x_min: float = -10, x_max: float = 10, step: float = 0.1):
        """Plot the polynomial."""
        x_values = np.arange(x_min, x_max + step, step)
        y_values = [self.evaluate(x) for x in x_values]
        
        plt.plot(x_values, y_values, label=str(self), linewidth=2)
        plt.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
        plt.axvline(x=0, color='k', linestyle='-', linewidth=0.5)
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.xlabel('x')
        plt.ylabel('f(x)')
        plt.title(f'Graph of f(x) = {self}')
        plt.show()
    
    def derivative(self):
        """Return the derivative using the power rule."""
        if self.degree() == 0:
            return Polynomial([0])
        
        new_coeffs = []
        for i in range(1, len(self.coeffs)):
            new_coeffs.append(i * self.coeffs[i])
        
        return Polynomial(new_coeffs)
    
    def integrate(self, C: float = 0):
        """Return the integral using the reverse power rule."""
        new_coeffs = [C]
        for i, coeff in enumerate(self.coeffs):
            new_coeffs.append(coeff / (i + 1))
        
        return Polynomial(new_coeffs)

In [ ]:
# Quick test
p = Polynomial([3, -4, 1])  # 3 - 4x + x²
print(f"Polynomial: {p}")
print(f"Degree: {p.degree()}")
print(f"At x=2: {p.evaluate(2)}")
print(f"Derivative: {p.derivative()}")

## Part B: Addition - Combining Like Terms

### The Mathematical Idea

When we add polynomials, we combine **like terms** (terms with the same power):

```
(2 + 3x + x²) + (1 + x + 2x²) = (2+1) + (3+1)x + (1+2)x²
                               = 3 + 4x + 3x²
```

In terms of coefficient lists:
```
[2, 3, 1] + [1, 1, 2] = [2+1, 3+1, 1+2] = [3, 4, 3]
```

**The challenge:** The lists might have different lengths!

```
(2 + 3x) + (1 + x²) = 2 + 3x + x²
[2, 3] + [1, 0, 1] = [2+1, 3+0, 0+1] = [3, 3, 1]
```

We need to handle the missing coefficients as zeros.

### YOUR TURN: Implement __add__()

The `__add__` method lets us use the `+` operator between polynomials.

**Hints:**
- Find the maximum length of the two coefficient lists
- Loop through indices up to that maximum
- For each index, get the coefficient from each polynomial (use 0 if index is out of bounds)
- Add the coefficients
- Return a new `Polynomial` with the sum coefficients

In [ ]:
class Polynomial:
    def __init__(self, coefficients: list):
        self.coeffs = coefficients
    
    def degree(self) -> int:
        for i in range(len(self.coeffs) - 1, -1, -1):
            if self.coeffs[i] != 0:
                return i
        return 0
    
    def evaluate(self, x: float) -> float:
        result = 0
        for i, coeff in enumerate(self.coeffs):
            result += coeff * (x ** i)
        return result
    
    def __str__(self) -> str:
        terms = []
        for i, coeff in enumerate(self.coeffs):
            if coeff == 0:
                continue
            if i == 0:
                terms.append(f"{coeff}")
            elif i == 1:
                terms.append(f"{coeff}x")
            else:
                terms.append(f"{coeff}x^{i}")
        if not terms:
            return "0"
        return " + ".join(terms)
    
    def plot(self, x_min: float = -10, x_max: float = 10, step: float = 0.1):
        x_values = np.arange(x_min, x_max + step, step)
        y_values = [self.evaluate(x) for x in x_values]
        
        plt.plot(x_values, y_values, label=str(self), linewidth=2)
        plt.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
        plt.axvline(x=0, color='k', linestyle='-', linewidth=0.5)
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.xlabel('x')
        plt.ylabel('f(x)')
        plt.title(f'Graph of f(x) = {self}')
        plt.show()
    
    def derivative(self):
        if self.degree() == 0:
            return Polynomial([0])
        
        new_coeffs = []
        for i in range(1, len(self.coeffs)):
            new_coeffs.append(i * self.coeffs[i])
        
        return Polynomial(new_coeffs)
    
    def integrate(self, C: float = 0):
        new_coeffs = [C]
        for i, coeff in enumerate(self.coeffs):
            new_coeffs.append(coeff / (i + 1))
        
        return Polynomial(new_coeffs)
    
    def __add__(self, other):
        """
        Add two polynomials by adding corresponding coefficients.
        
        Parameters:
        -----------
        other : Polynomial
            The polynomial to add
            
        Returns:
        --------
        Polynomial
            A new polynomial representing the sum
        """
        # YOUR CODE HERE
        pass

In [ ]:
# Test your implementation
p1 = Polynomial([2, 3, 1])      # 2 + 3x + x²
p2 = Polynomial([1, 1, 0, 2])   # 1 + x + 2x³

p3 = p1 + p2
print(f"({p1}) + ({p2})")
print(f"= {p3}")

# Verify by evaluating
x = 5
print(f"\nAt x={x}:")
print(f"p1({x}) = {p1.evaluate(x)}")
print(f"p2({x}) = {p2.evaluate(x)}")
print(f"p1({x}) + p2({x}) = {p1.evaluate(x) + p2.evaluate(x)}")
print(f"p3({x}) = {p3.evaluate(x)}")
print(f"Match? {abs(p3.evaluate(x) - (p1.evaluate(x) + p2.evaluate(x))) < 1e-10}")

**Solution:**

In [ ]:
class Polynomial:
    def __init__(self, coefficients: list):
        self.coeffs = coefficients
    
    def degree(self) -> int:
        for i in range(len(self.coeffs) - 1, -1, -1):
            if self.coeffs[i] != 0:
                return i
        return 0
    
    def evaluate(self, x: float) -> float:
        result = 0
        for i, coeff in enumerate(self.coeffs):
            result += coeff * (x ** i)
        return result
    
    def __str__(self) -> str:
        terms = []
        for i, coeff in enumerate(self.coeffs):
            if coeff == 0:
                continue
            if i == 0:
                terms.append(f"{coeff}")
            elif i == 1:
                terms.append(f"{coeff}x")
            else:
                terms.append(f"{coeff}x^{i}")
        if not terms:
            return "0"
        return " + ".join(terms)
    
    def plot(self, x_min: float = -10, x_max: float = 10, step: float = 0.1):
        x_values = np.arange(x_min, x_max + step, step)
        y_values = [self.evaluate(x) for x in x_values]
        
        plt.plot(x_values, y_values, label=str(self), linewidth=2)
        plt.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
        plt.axvline(x=0, color='k', linestyle='-', linewidth=0.5)
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.xlabel('x')
        plt.ylabel('f(x)')
        plt.title(f'Graph of f(x) = {self}')
        plt.show()
    
    def derivative(self):
        if self.degree() == 0:
            return Polynomial([0])
        
        new_coeffs = []
        for i in range(1, len(self.coeffs)):
            new_coeffs.append(i * self.coeffs[i])
        
        return Polynomial(new_coeffs)
    
    def integrate(self, C: float = 0):
        new_coeffs = [C]
        for i, coeff in enumerate(self.coeffs):
            new_coeffs.append(coeff / (i + 1))
        
        return Polynomial(new_coeffs)
    
    def __add__(self, other):
        """Add two polynomials."""
        # Find the maximum length
        len1 = len(self.coeffs)
        len2 = len(other.coeffs)
        max_len = max(len1, len2)
        
        # Add corresponding coefficients
        new_coeffs = []
        for i in range(max_len):
            c1 = self.coeffs[i] if i < len1 else 0
            c2 = other.coeffs[i] if i < len2 else 0
            new_coeffs.append(c1 + c2)
        
        return Polynomial(new_coeffs)

## Exploring Addition

In [ ]:
p1 = Polynomial([2, 3, 1])      # 2 + 3x + x²
p2 = Polynomial([1, 1, 0, 2])   # 1 + x + 2x³

p3 = p1 + p2
print(f"({p1}) + ({p2}) = {p3}")

In [ ]:
# What if we add a polynomial to itself?
p = Polynomial([1, 2, 3])
print(f"p = {p}")

double = p + p
print(f"p + p = {double}")
# All coefficients should be doubled!

In [ ]:
# What about adding opposites?
p1 = Polynomial([1, 2, 3])
p2 = Polynomial([-1, -2, -3])

zero = p1 + p2
print(f"({p1}) + ({p2}) = {zero}")
# Should be the zero polynomial!

## Part C: Subtraction - The Opposite Operation

Subtraction is very similar to addition - we just subtract corresponding coefficients instead of adding them.

### YOUR TURN: Implement __sub__()

In [ ]:
class Polynomial:
    def __init__(self, coefficients: list):
        self.coeffs = coefficients
    
    def degree(self) -> int:
        for i in range(len(self.coeffs) - 1, -1, -1):
            if self.coeffs[i] != 0:
                return i
        return 0
    
    def evaluate(self, x: float) -> float:
        result = 0
        for i, coeff in enumerate(self.coeffs):
            result += coeff * (x ** i)
        return result
    
    def __str__(self) -> str:
        terms = []
        for i, coeff in enumerate(self.coeffs):
            if coeff == 0:
                continue
            if i == 0:
                terms.append(f"{coeff}")
            elif i == 1:
                terms.append(f"{coeff}x")
            else:
                terms.append(f"{coeff}x^{i}")
        if not terms:
            return "0"
        return " + ".join(terms)
    
    def plot(self, x_min: float = -10, x_max: float = 10, step: float = 0.1):
        x_values = np.arange(x_min, x_max + step, step)
        y_values = [self.evaluate(x) for x in x_values]
        
        plt.plot(x_values, y_values, label=str(self), linewidth=2)
        plt.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
        plt.axvline(x=0, color='k', linestyle='-', linewidth=0.5)
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.xlabel('x')
        plt.ylabel('f(x)')
        plt.title(f'Graph of f(x) = {self}')
        plt.show()
    
    def derivative(self):
        if self.degree() == 0:
            return Polynomial([0])
        
        new_coeffs = []
        for i in range(1, len(self.coeffs)):
            new_coeffs.append(i * self.coeffs[i])
        
        return Polynomial(new_coeffs)
    
    def integrate(self, C: float = 0):
        new_coeffs = [C]
        for i, coeff in enumerate(self.coeffs):
            new_coeffs.append(coeff / (i + 1))
        
        return Polynomial(new_coeffs)
    
    def __add__(self, other):
        len1 = len(self.coeffs)
        len2 = len(other.coeffs)
        max_len = max(len1, len2)
        
        new_coeffs = []
        for i in range(max_len):
            c1 = self.coeffs[i] if i < len1 else 0
            c2 = other.coeffs[i] if i < len2 else 0
            new_coeffs.append(c1 + c2)
        
        return Polynomial(new_coeffs)
    
    def __sub__(self, other):
        """
        Subtract two polynomials by subtracting corresponding coefficients.
        
        Parameters:
        -----------
        other : Polynomial
            The polynomial to subtract
            
        Returns:
        --------
        Polynomial
            A new polynomial representing the difference
        """
        # YOUR CODE HERE
        pass

In [ ]:
# Test your implementation
p1 = Polynomial([5, 3, 1])
p2 = Polynomial([2, 1, 0, 2])

p3 = p1 - p2
print(f"({p1}) - ({p2}) = {p3}")

# Verify
x = 3
print(f"\nAt x={x}:")
print(f"p1({x}) - p2({x}) = {p1.evaluate(x) - p2.evaluate(x)}")
print(f"p3({x}) = {p3.evaluate(x)}")

**Solution:**

In [ ]:
# Just change + to - in the __add__ method!
def __sub__(self, other):
    """Subtract two polynomials."""
    len1 = len(self.coeffs)
    len2 = len(other.coeffs)
    max_len = max(len1, len2)
    
    new_coeffs = []
    for i in range(max_len):
        c1 = self.coeffs[i] if i < len1 else 0
        c2 = other.coeffs[i] if i < len2 else 0
        new_coeffs.append(c1 - c2)  # Subtract instead of add!
    
    return Polynomial(new_coeffs)

Let's add this to our complete class:

In [ ]:
class Polynomial:
    def __init__(self, coefficients: list):
        self.coeffs = coefficients
    
    def degree(self) -> int:
        for i in range(len(self.coeffs) - 1, -1, -1):
            if self.coeffs[i] != 0:
                return i
        return 0
    
    def evaluate(self, x: float) -> float:
        result = 0
        for i, coeff in enumerate(self.coeffs):
            result += coeff * (x ** i)
        return result
    
    def __str__(self) -> str:
        terms = []
        for i, coeff in enumerate(self.coeffs):
            if coeff == 0:
                continue
            if i == 0:
                terms.append(f"{coeff}")
            elif i == 1:
                terms.append(f"{coeff}x")
            else:
                terms.append(f"{coeff}x^{i}")
        if not terms:
            return "0"
        return " + ".join(terms)
    
    def plot(self, x_min: float = -10, x_max: float = 10, step: float = 0.1):
        x_values = np.arange(x_min, x_max + step, step)
        y_values = [self.evaluate(x) for x in x_values]
        
        plt.plot(x_values, y_values, label=str(self), linewidth=2)
        plt.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
        plt.axvline(x=0, color='k', linestyle='-', linewidth=0.5)
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.xlabel('x')
        plt.ylabel('f(x)')
        plt.title(f'Graph of f(x) = {self}')
        plt.show()
    
    def derivative(self):
        if self.degree() == 0:
            return Polynomial([0])
        
        new_coeffs = []
        for i in range(1, len(self.coeffs)):
            new_coeffs.append(i * self.coeffs[i])
        
        return Polynomial(new_coeffs)
    
    def integrate(self, C: float = 0):
        new_coeffs = [C]
        for i, coeff in enumerate(self.coeffs):
            new_coeffs.append(coeff / (i + 1))
        
        return Polynomial(new_coeffs)
    
    def __add__(self, other):
        len1 = len(self.coeffs)
        len2 = len(other.coeffs)
        max_len = max(len1, len2)
        
        new_coeffs = []
        for i in range(max_len):
            c1 = self.coeffs[i] if i < len1 else 0
            c2 = other.coeffs[i] if i < len2 else 0
            new_coeffs.append(c1 + c2)
        
        return Polynomial(new_coeffs)
    
    def __sub__(self, other):
        len1 = len(self.coeffs)
        len2 = len(other.coeffs)
        max_len = max(len1, len2)
        
        new_coeffs = []
        for i in range(max_len):
            c1 = self.coeffs[i] if i < len1 else 0
            c2 = other.coeffs[i] if i < len2 else 0
            new_coeffs.append(c1 - c2)
        
        return Polynomial(new_coeffs)

In [ ]:
# Test subtraction
p1 = Polynomial([5, 3, 1])
p2 = Polynomial([2, 1])

print(f"p1 = {p1}")
print(f"p2 = {p2}")
print(f"p1 - p2 = {p1 - p2}")
print(f"p2 - p1 = {p2 - p1}")

In [ ]:
# Subtract from itself - should get zero
p = Polynomial([1, 2, 3, 4])
zero = p - p
print(f"({p}) - ({p}) = {zero}")

## Part D: Multiplication - A More Complex Operation

### The Mathematical Idea

When we multiply polynomials, **each term in the first** multiplies **each term in the second**:

```
(2 + x)(3 + x) = 2·3 + 2·x + x·3 + x·x
               = 6 + 2x + 3x + x²
               = 6 + 5x + x²
```

More systematically, using the distributive property:
```
(c₀ + c₁x)(d₀ + d₁x) = c₀d₀ + c₀d₁x + c₁d₀x + c₁d₁x²
                      = c₀d₀ + (c₀d₁ + c₁d₀)x + c₁d₁x²
```

**Key insight:** When we multiply `cᵢxⁱ` by `dⱼxʲ`, we get `cᵢdⱼx^(i+j)`

**Pattern:** 
- The degree of the product is the sum of the degrees
- `result[i+j] += coeffs1[i] * coeffs2[j]`

### YOUR TURN: Implement __mul__()

**Hints:**
- Create a result list with length `degree(p1) + degree(p2) + 1`
- Initialize all coefficients to 0
- Use nested loops: for each i in first polynomial, for each j in second polynomial
- Add `coeffs1[i] * coeffs2[j]` to `result[i+j]`

In [ ]:
class Polynomial:
    # ... (previous methods) ...
    
    def __mul__(self, other):
        """
        Multiply two polynomials.
        
        Each term in self multiplies each term in other:
        (Σ aᵢxⁱ) * (Σ bⱼxʲ) = Σᵢ Σⱼ aᵢbⱼx^(i+j)
        
        Parameters:
        -----------
        other : Polynomial
            The polynomial to multiply by
            
        Returns:
        --------
        Polynomial
            A new polynomial representing the product
        """
        # YOUR CODE HERE
        pass

**Think about it first:**

If we multiply:
- A polynomial of degree 1 (linear)
- By another polynomial of degree 1 (linear)
- What degree should the result be?

```python
# (1 + x)(1 + x) = 1 + x + x + x² = 1 + 2x + x²
# Degree 1 * degree 1 = degree 2
```

In [ ]:
# Test once you've implemented it
p1 = Polynomial([1, 1])  # 1 + x
p2 = Polynomial([1, 1])  # 1 + x

product = p1 * p2
print(f"({p1}) * ({p2}) = {product}")
# Should be 1 + 2x + x²

**Solution:**

In [ ]:
def __mul__(self, other):
    """Multiply two polynomials."""
    # Result has degree = degree(self) + degree(other)
    result_degree = self.degree() + other.degree()
    new_coeffs = [0] * (result_degree + 1)
    
    # Multiply each term in self by each term in other
    for i, coeff_a in enumerate(self.coeffs):
        for j, coeff_b in enumerate(other.coeffs):
            # Term aᵢxⁱ * bⱼxʲ = aᵢbⱼx^(i+j)
            power = i + j
            new_coeffs[power] += coeff_a * coeff_b
    
    return Polynomial(new_coeffs)

Let's add multiplication to our complete class and test it:

In [ ]:
class Polynomial:
    def __init__(self, coefficients: list):
        self.coeffs = coefficients
    
    def degree(self) -> int:
        for i in range(len(self.coeffs) - 1, -1, -1):
            if self.coeffs[i] != 0:
                return i
        return 0
    
    def evaluate(self, x: float) -> float:
        result = 0
        for i, coeff in enumerate(self.coeffs):
            result += coeff * (x ** i)
        return result
    
    def __str__(self) -> str:
        terms = []
        for i, coeff in enumerate(self.coeffs):
            if coeff == 0:
                continue
            if i == 0:
                terms.append(f"{coeff}")
            elif i == 1:
                terms.append(f"{coeff}x")
            else:
                terms.append(f"{coeff}x^{i}")
        if not terms:
            return "0"
        return " + ".join(terms)
    
    def plot(self, x_min: float = -10, x_max: float = 10, step: float = 0.1):
        x_values = np.arange(x_min, x_max + step, step)
        y_values = [self.evaluate(x) for x in x_values]
        
        plt.plot(x_values, y_values, label=str(self), linewidth=2)
        plt.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
        plt.axvline(x=0, color='k', linestyle='-', linewidth=0.5)
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.xlabel('x')
        plt.ylabel('f(x)')
        plt.title(f'Graph of f(x) = {self}')
        plt.show()
    
    def derivative(self):
        if self.degree() == 0:
            return Polynomial([0])
        
        new_coeffs = []
        for i in range(1, len(self.coeffs)):
            new_coeffs.append(i * self.coeffs[i])
        
        return Polynomial(new_coeffs)
    
    def integrate(self, C: float = 0):
        new_coeffs = [C]
        for i, coeff in enumerate(self.coeffs):
            new_coeffs.append(coeff / (i + 1))
        
        return Polynomial(new_coeffs)
    
    def __add__(self, other):
        len1 = len(self.coeffs)
        len2 = len(other.coeffs)
        max_len = max(len1, len2)
        
        new_coeffs = []
        for i in range(max_len):
            c1 = self.coeffs[i] if i < len1 else 0
            c2 = other.coeffs[i] if i < len2 else 0
            new_coeffs.append(c1 + c2)
        
        return Polynomial(new_coeffs)
    
    def __sub__(self, other):
        len1 = len(self.coeffs)
        len2 = len(other.coeffs)
        max_len = max(len1, len2)
        
        new_coeffs = []
        for i in range(max_len):
            c1 = self.coeffs[i] if i < len1 else 0
            c2 = other.coeffs[i] if i < len2 else 0
            new_coeffs.append(c1 - c2)
        
        return Polynomial(new_coeffs)
    
    def __mul__(self, other):
        result_degree = self.degree() + other.degree()
        new_coeffs = [0] * (result_degree + 1)
        
        for i, coeff_a in enumerate(self.coeffs):
            for j, coeff_b in enumerate(other.coeffs):
                power = i + j
                new_coeffs[power] += coeff_a * coeff_b
        
        return Polynomial(new_coeffs)

## Exploring Multiplication

In [ ]:
# Simple case: (1 + x)(1 + x)
p = Polynomial([1, 1])
p_squared = p * p
print(f"({p})² = {p_squared}")
# Should be 1 + 2x + x²

In [ ]:
# Verify a factorization: x² - 1 = (x + 1)(x - 1)
p1 = Polynomial([1, 1])    # 1 + x (which is x + 1)
p2 = Polynomial([-1, 1])   # -1 + x (which is x - 1)

product = p1 * p2
print(f"({p1}) * ({p2}) = {product}")

expected = Polynomial([-1, 0, 1])  # -1 + 0x + x²
print(f"Expected: {expected}")

In [ ]:
# Build higher powers
p = Polynomial([1, 1])  # 1 + x

print(f"(1 + x)⁰ = 1")
print(f"(1 + x)¹ = {p}")
print(f"(1 + x)² = {p * p}")
print(f"(1 + x)³ = {p * p * p}")
print(f"(1 + x)⁴ = {p * p * p * p}")

# Notice the coefficients - they're binomial coefficients!
# Pascal's triangle!

In [ ]:
# Verify by evaluation
p1 = Polynomial([2, 3])
p2 = Polynomial([1, -1])
product = p1 * p2

x = 7
print(f"At x={x}:")
print(f"p1({x}) = {p1.evaluate(x)}")
print(f"p2({x}) = {p2.evaluate(x)}")
print(f"p1({x}) * p2({x}) = {p1.evaluate(x) * p2.evaluate(x)}")
print(f"product({x}) = {product.evaluate(x)}")
print(f"Match? {abs(product.evaluate(x) - p1.evaluate(x) * p2.evaluate(x)) < 1e-10}")

## Part E: Testing Equality

It's useful to check if two polynomials are the same. But there's a subtlety:

```python
[1, 2, 0] and [1, 2] represent the same polynomial (1 + 2x)
```

We need to compare them based on their **actual degrees**, not just list equality.

### Implementing __eq__()

In [ ]:
class Polynomial:
    # ... all previous methods ...
    
    def __eq__(self, other):
        """
        Test if two polynomials are equal.
        
        Careful: [1, 2, 0] and [1, 2] represent the same polynomial!
        
        Parameters:
        -----------
        other : Polynomial
            The polynomial to compare to
            
        Returns:
        --------
        bool
            True if polynomials are equal
        """
        # First check degrees
        d1 = self.degree()
        d2 = other.degree()
        
        if d1 != d2:
            return False
        
        # Compare coefficients up to the degree
        for i in range(d1 + 1):
            c1 = self.coeffs[i] if i < len(self.coeffs) else 0
            c2 = other.coeffs[i] if i < len(other.coeffs) else 0
            # Allow tiny numerical errors
            if abs(c1 - c2) > 1e-10:
                return False
        
        return True

Let's add this to our complete class and use it to verify algebraic identities:

In [ ]:
class Polynomial:
    def __init__(self, coefficients: list):
        self.coeffs = coefficients
    
    def degree(self) -> int:
        for i in range(len(self.coeffs) - 1, -1, -1):
            if self.coeffs[i] != 0:
                return i
        return 0
    
    def evaluate(self, x: float) -> float:
        result = 0
        for i, coeff in enumerate(self.coeffs):
            result += coeff * (x ** i)
        return result
    
    def __str__(self) -> str:
        terms = []
        for i, coeff in enumerate(self.coeffs):
            if coeff == 0:
                continue
            if i == 0:
                terms.append(f"{coeff}")
            elif i == 1:
                terms.append(f"{coeff}x")
            else:
                terms.append(f"{coeff}x^{i}")
        if not terms:
            return "0"
        return " + ".join(terms)
    
    def plot(self, x_min: float = -10, x_max: float = 10, step: float = 0.1):
        x_values = np.arange(x_min, x_max + step, step)
        y_values = [self.evaluate(x) for x in x_values]
        
        plt.plot(x_values, y_values, label=str(self), linewidth=2)
        plt.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
        plt.axvline(x=0, color='k', linestyle='-', linewidth=0.5)
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.xlabel('x')
        plt.ylabel('f(x)')
        plt.title(f'Graph of f(x) = {self}')
        plt.show()
    
    def derivative(self):
        if self.degree() == 0:
            return Polynomial([0])
        
        new_coeffs = []
        for i in range(1, len(self.coeffs)):
            new_coeffs.append(i * self.coeffs[i])
        
        return Polynomial(new_coeffs)
    
    def integrate(self, C: float = 0):
        new_coeffs = [C]
        for i, coeff in enumerate(self.coeffs):
            new_coeffs.append(coeff / (i + 1))
        
        return Polynomial(new_coeffs)
    
    def __add__(self, other):
        len1 = len(self.coeffs)
        len2 = len(other.coeffs)
        max_len = max(len1, len2)
        
        new_coeffs = []
        for i in range(max_len):
            c1 = self.coeffs[i] if i < len1 else 0
            c2 = other.coeffs[i] if i < len2 else 0
            new_coeffs.append(c1 + c2)
        
        return Polynomial(new_coeffs)
    
    def __sub__(self, other):
        len1 = len(self.coeffs)
        len2 = len(other.coeffs)
        max_len = max(len1, len2)
        
        new_coeffs = []
        for i in range(max_len):
            c1 = self.coeffs[i] if i < len1 else 0
            c2 = other.coeffs[i] if i < len2 else 0
            new_coeffs.append(c1 - c2)
        
        return Polynomial(new_coeffs)
    
    def __mul__(self, other):
        result_degree = self.degree() + other.degree()
        new_coeffs = [0] * (result_degree + 1)
        
        for i, coeff_a in enumerate(self.coeffs):
            for j, coeff_b in enumerate(other.coeffs):
                power = i + j
                new_coeffs[power] += coeff_a * coeff_b
        
        return Polynomial(new_coeffs)
    
    def __eq__(self, other):
        d1 = self.degree()
        d2 = other.degree()
        
        if d1 != d2:
            return False
        
        for i in range(d1 + 1):
            c1 = self.coeffs[i] if i < len(self.coeffs) else 0
            c2 = other.coeffs[i] if i < len(other.coeffs) else 0
            if abs(c1 - c2) > 1e-10:
                return False
        
        return True

## Part F: Verifying Algebraic Identities

Now we can use our polynomial arithmetic to verify famous algebraic identities!

In [ ]:
# Identity: (a + b)² = a² + 2ab + b²
a = Polynomial([0, 1])  # x
b = Polynomial([1])     # 1

lhs = (a + b) * (a + b)
rhs = a*a + Polynomial([2])*a*b + b*b

print(f"(a + b)² = {lhs}")
print(f"a² + 2ab + b² = {rhs}")
print(f"Equal? {lhs == rhs}")

In [ ]:
# Identity: (a + b)(a - b) = a² - b²
a = Polynomial([0, 1])  # x
b = Polynomial([1])     # 1

lhs = (a + b) * (a - b)
rhs = a*a - b*b

print(f"(a + b)(a - b) = {lhs}")
print(f"a² - b² = {rhs}")
print(f"Equal? {lhs == rhs}")

In [ ]:
# Identity: (a + b)³ = a³ + 3a²b + 3ab² + b³
a = Polynomial([0, 1])  # x
b = Polynomial([1])     # 1

lhs = (a + b) * (a + b) * (a + b)
rhs = (a*a*a + 
       Polynomial([3])*a*a*b + 
       Polynomial([3])*a*b*b + 
       b*b*b)

print(f"(a + b)³ = {lhs}")
print(f"a³ + 3a²b + 3ab² + b³ = {rhs}")
print(f"Equal? {lhs == rhs}")

## Part G: Building Polynomials from Roots

If we know the roots of a polynomial, we can build it by multiplying linear factors:

In [ ]:
def polynomial_from_roots(roots: list):
    """
    Build a polynomial with given roots.
    
    If roots are r₁, r₂, r₃, then polynomial is:
    (x - r₁)(x - r₂)(x - r₃)
    
    Parameters:
    -----------
    roots : list
        List of roots
        
    Returns:
    --------
    Polynomial
        The polynomial with those roots
    """
    result = Polynomial([1])  # Start with 1
    
    for root in roots:
        # Multiply by (x - root)
        factor = Polynomial([-root, 1])  # -root + x
        result = result * factor
    
    return result

In [ ]:
# Build a polynomial with roots at 1, 2, 3
p = polynomial_from_roots([1, 2, 3])
print(f"Polynomial with roots 1, 2, 3: {p}")

# Verify the roots
for root in [1, 2, 3]:
    print(f"p({root}) = {p.evaluate(root)}")

# Expand manually to check:
# (x-1)(x-2)(x-3) = (x-1)(x² - 5x + 6)
#                  = x³ - 5x² + 6x - x² + 5x - 6
#                  = x³ - 6x² + 11x - 6
# Coefficients: [-6, 11, -6, 1] ✓

In [ ]:
# Build and plot a polynomial with roots at -2, 0, 1, 3
p = polynomial_from_roots([-2, 0, 1, 3])
print(f"Polynomial: {p}")
p.plot(x_min=-3, x_max=4)

## Summary: The Power of Polynomial Arithmetic

We've now implemented:

### Operations:
- **Addition** (`+`): Combine like terms
- **Subtraction** (`-`): Subtract coefficients
- **Multiplication** (`*`): Each term multiplies each term
- **Equality** (`==`): Compare degrees and coefficients

### From Tutorial 3:
- **Derivatives**: Power rule
- **Integrals**: Reverse power rule
- **Evaluation**: Calculate f(x)
- **Plotting**: Visualize

### What we can do now:
- Verify algebraic identities
- Build polynomials from roots
- Explore binomial expansions
- Factor and expand
- Work with polynomials of ANY degree!

## Exercises

### Exercise 1: Binomial Coefficients

In [ ]:
# Compute (1 + x)^n for n = 0, 1, 2, 3, 4, 5
# Print each result
# Notice the coefficients - do they form a pattern?
# This is Pascal's triangle!

# YOUR CODE HERE

### Exercise 2: Factoring Check

In [ ]:
# Verify these factorizations by multiplying:
# 1. x² - 4 = (x + 2)(x - 2)
# 2. x² - 5x + 6 = (x - 2)(x - 3)
# 3. x³ - 1 = (x - 1)(x² + x + 1)

# YOUR CODE HERE

### Exercise 3: Building from Roots

In [ ]:
# Use polynomial_from_roots to build polynomials with:
# 1. Roots at -1 and 1
# 2. Roots at 0, 0, and 2 (a repeated root!)
# 3. Roots at -2, -1, 1, 2

# For each one:
# - Print the polynomial
# - Verify the roots by evaluation
# - Plot it

# YOUR CODE HERE

### Exercise 4: Derivative of a Product

In [ ]:
# The product rule says: (fg)' = f'g + fg'

# Let f(x) = x² and g(x) = x + 1
# Compute (fg)' two ways:
# 1. Multiply f and g, then take derivative
# 2. Use product rule: f'g + fg'

# Do they match?

# YOUR CODE HERE

### Exercise 5: Working with Differences

In [ ]:
# Create p(x) = x³ - 3x² + 3x - 1
# Create q(x) = x³

# Compute p - q
# What do you get?

# Now try: is (x - 1)³ equal to p?
# Hint: Use multiplication to expand (x - 1)³

# YOUR CODE HERE

## Reflection and Looking Ahead

Our `Polynomial` class is now very powerful! We can:
- Represent any polynomial
- Add, subtract, multiply them
- Take derivatives and integrals
- Verify algebraic identities
- Build polynomials from roots

### But we still have the trade-off from Tutorial 3:

**What we gained:**
- ✓ Generality (any degree)
- ✓ Rich operations (+, -, *, derivative, integral)
- ✓ No code duplication
- ✓ Clean, elegant implementation

**What we lost:**
- ✗ Easy exact root formulas (quadratic formula)
- ✗ Special methods (vertex, discriminant)
- ✗ Intuitive interface for common cases
- ✗ Clear names (have to remember coefficient order)

### Next Time: Tutorial 5 - Inheritance

In Tutorial 5, we'll learn about **inheritance** - a way to create specialized `Linear` and `Quadratic` classes that:
- **Inherit** all the general methods from `Polynomial` (derivative, integrate, +, -, *, etc.)
- **Add** specialized methods (vertex, discriminant, exact root formulas)
- **Override** methods where needed (prettier `__str__`)
- **Work together** seamlessly

This will give us **the best of both worlds**!

---

**End of Tutorial 4**